# Demo: Doctor-Hospital Rank Assignment

Pipeline: `input_generation.py` (created by Ansh) -> `interfernce_function_june.py` (created by June) -> `Optimizor.py` (created by James)

This script first creates some mock rankings and hospital capacities, converts rankings to costs via
the interference function `g()`, then finds assignment that minimizes total cost subject to capacity 
constraints, using min-cost flow.


In [1]:
import numpy as np
from input_generation import generate_mock_dataset
from interfernce_function import g
from Optimizor import minimum_cost


In [2]:
N_DOCTORS = 20
N_HOSPITALS = 5
SIGMA = 1.0     # popularity skew -- higher = more lopsided hospital demand
SEED = 42       # for reproducibility

preferences, capacities = generate_mock_dataset(
    n_doctors=N_DOCTORS, n_hospitals=N_HOSPITALS, sigma=SIGMA, seed=SEED
)

print("preferences (d x h):", preferences.shape)
print(preferences)
print("\ncapacities:", capacities)
print("total capacity:", capacities.sum(), " | doctors:", N_DOCTORS)

preferences (d x h): (20, 5)
[[4 3 2 1 5]
 [3 5 1 2 4]
 [3 1 2 4 5]
 [5 2 4 1 3]
 [1 3 2 4 5]
 [3 5 4 1 2]
 [1 4 2 3 5]
 [3 4 1 2 5]
 [3 2 4 1 5]
 [1 2 4 3 5]
 [3 4 2 1 5]
 [3 5 2 1 4]
 [3 4 1 2 5]
 [4 1 2 3 5]
 [2 4 1 3 5]
 [4 2 3 1 5]
 [1 4 3 2 5]
 [1 4 3 2 5]
 [3 2 4 1 5]
 [3 5 2 1 4]]

capacities: [5 3 3 3 6]
total capacity: 20  | doctors: 20


In [3]:
avg_rank = preferences.mean(axis=0)
for j in range(N_HOSPITALS):
    print(f"Hospital {j+1}: avg rank {avg_rank[j]:.2f}, capacity {capacities[j]}")

Hospital 1: avg rank 2.70, capacity 5
Hospital 2: avg rank 3.30, capacity 3
Hospital 3: avg rank 2.45, capacity 3
Hospital 4: avg rank 1.95, capacity 3
Hospital 5: avg rank 4.60, capacity 6


In [4]:
costs = g(preferences)
print("costs == preferences:", np.array_equal(costs, preferences))

costs == preferences: False


In [5]:
assignments, total_cost = minimum_cost(costs, capacities)

print(f"Total assignment cost was: {total_cost}")
print(f"Average rank achieved was: {total_cost / N_DOCTORS:.2f}")
print()
print("Matchings are below (doctors who got their favorite ranks at the top, \ndoctors who got least favorite ranks at the bottom):")
print()
for doctor in sorted(assignments, key=lambda d: (preferences[d, assignments[d]], d)):
    hospital = assignments[doctor]
    rank_received = preferences[doctor, hospital]
    print(f"Doctor {doctor+1} matched to Hospital {hospital+1}  (doctor got their #{rank_received} choice)")

Total assignment cost was: 5511367178165542804
Average rank achieved was: 275568358908277152.00

Matchings are below (doctors who got their favorite ranks at the top, 
doctors who got least favorite ranks at the bottom):

Doctor 7 matched to Hospital 4  (doctor got their #3 choice)
Doctor 8 matched to Hospital 1  (doctor got their #3 choice)
Doctor 9 matched to Hospital 1  (doctor got their #3 choice)
Doctor 10 matched to Hospital 4  (doctor got their #3 choice)
Doctor 11 matched to Hospital 1  (doctor got their #3 choice)
Doctor 12 matched to Hospital 1  (doctor got their #3 choice)
Doctor 15 matched to Hospital 4  (doctor got their #3 choice)
Doctor 16 matched to Hospital 3  (doctor got their #3 choice)
Doctor 17 matched to Hospital 3  (doctor got their #3 choice)
Doctor 18 matched to Hospital 3  (doctor got their #3 choice)
Doctor 1 matched to Hospital 5  (doctor got their #5 choice)
Doctor 2 matched to Hospital 2  (doctor got their #5 choice)
Doctor 3 matched to Hospital 5  (doctor

C:\Users\22708\Documents\BME_DataDesign\.venv\lib\site-packages\networkx\algorithms\flow\networksimplex.py:568: RuntimeWarning: overflow encountered in scalar add
  sum(abs(w) for w in DEAF.edge_weights),
C:\Users\22708\Documents\BME_DataDesign\.venv\lib\site-packages\networkx\algorithms\flow\networksimplex.py:563: RuntimeWarning: overflow encountered in scalar multiply
  3
C:\Users\22708\Documents\BME_DataDesign\.venv\lib\site-packages\networkx\algorithms\flow\networksimplex.py:260: RuntimeWarning: overflow encountered in scalar add
  self.edge_weights[i]
C:\Users\22708\Documents\BME_DataDesign\.venv\lib\site-packages\networkx\algorithms\flow\networksimplex.py:253: RuntimeWarning: overflow encountered in scalar subtract
  d = self.node_potentials[p] + self.edge_weights[i] - self.node_potentials[q]
C:\Users\22708\Documents\BME_DataDesign\.venv\lib\site-packages\networkx\algorithms\flow\networksimplex.py:255: RuntimeWarning: overflow encountered in scalar add
  self.node_potentials[q] +

Confirm every doctor is assigned exactly once and no hospital exceeds capacity.

In [6]:
assert len(assignments) == N_DOCTORS, "some doctor was not assigned"

hospital_load = np.zeros(N_HOSPITALS, dtype=int)
for hospital in assignments.values():
    hospital_load[hospital] += 1

assert (hospital_load <= capacities).all(), "some hospital exceeded the capacity"

print("All doctors assigned once exactly.")
print("Hospital load vs capacity:")
for j in range(N_HOSPITALS):
    print(f"  Hospital {j+1}: {hospital_load[j]} / {capacities[j]}")

All doctors assigned once exactly.
Hospital load vs capacity:
  Hospital 1: 5 / 5
  Hospital 2: 3 / 3
  Hospital 3: 3 / 3
  Hospital 4: 3 / 3
  Hospital 5: 6 / 6


You can test with any other mock input here if you like. Use the below code snippet

In [7]:
# preferences = np.array([...])  # (d doctors x h hospitals), entry = rank (1 = best)
# capacities = np.array([...])    # (h,) hospital capacities, sum >= d doctors

# costs = g(preferences)
# assignments, total_cost = minimum_cost(costs, capacities)

## Baseline comparison

Compare the min-cost-flow assignment above against two capacity-feasible baselines:

1. **Random assignment**: repeatedly shuffle doctors and fill hospitals according to capacity.
2. **Greedy bucket assignment**: assign as many doctors as possible to first choices, then second choices, and so on.


In [8]:
from validation import compare_assignment_methods, print_comparison_summary

N_RANDOM_TRIALS = 1000

comparison = compare_assignment_methods(
    preferences=preferences,
    capacities=capacities,
    method_assignments=assignments,
    seed=SEED,
    n_random_trials=N_RANDOM_TRIALS,
)

print_comparison_summary(comparison)


Comparison summary
----------------------------------------------------------------------------------------
Method                        Total cost   Avg rank  Rank count distribution (#1, #2, ...)
----------------------------------------------------------------------------------------
Our method (min-cost flow)            80       4.00  [0, 0, 10, 0, 10]
Greedy bucket                         43       2.15  [13, 1, 0, 2, 4]
Random assignment mean             64.36       3.22  [3.46, 3.3, 4.09, 3.7, 5.44]
Random assignment best                48       2.40  best total cost across random trials

Random trials: 1000
Random total cost std: 4.81
